# Inverse EMA parameter sweep

Scan a 2-D grid of `ema_fast` × `ema_slow` values against the **inverse** EMA crossover strategy (fades the cross instead of riding it), compare the results, and visualise where the robust regions are.

**What this notebook produces**
- A table with one row per `(ema_fast, ema_slow)` combination
- Performance metrics (trades, win rate, PnL, Sharpe, drawdown, profit factor)
- Exit-reason breakdown per config (how many trades closed on SL / TP / trailing / etc.)
- Average trade duration per config
- Heatmaps so you can see the *region* of good params, not just the single peak

In [1]:
# --- path bootstrap: make 'entry_exit_points' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "entry_exit_points" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
import dataclasses
from collections import Counter

import pandas as pd
import plotly.express as px

from entry_exit_points.fetcher import BybitFetcher
from entry_exit_points.backtester import Backtester
from entry_exit_points.models import StrategyConfig, ExitReason
from entry_exit_points.strategies import InverseEMACrossoverStrategy

## 1. Data loader

`start_time` / `end_time` accept anything `pandas.Timestamp` parses (ISO strings, datetimes). Naive values are treated as UTC. Leave `END_TIME = None` to fetch up to *now*.

In [3]:
SYMBOL     = "BTCUSDT"
INTERVAL   = "15"
START_TIME = "2026-03-20"
END_TIME   = "2026-04-19"  # or None for "up to now"

In [4]:
fetcher = BybitFetcher()
df = fetcher.fetch_klines(
    symbol=SYMBOL,
    interval=INTERVAL,
    start_time=START_TIME,
    end_time=END_TIME,
)
fetcher.close()
print(f"Loaded {len(df)} bars from {df.index[0]} to {df.index[-1]}")

Loaded 2881 bars from 2026-03-20 00:00:00+00:00 to 2026-04-19 00:00:00+00:00


## 2. Sweep ranges

The full grid is evaluated — `fast >= slow` combinations are **not** skipped. When `fast == slow` the two EMAs are identical and no crosses fire (zero trades). When `fast > slow` the roles invert (the nominally 'fast' EMA is smoother); the strategy still trades, but on a different signal.

In [5]:
EMA_FAST_VALUES = range(5, 31, 2)     # 5, 7, 9, ..., 29
EMA_SLOW_VALUES = range(20, 101, 5)   # 20, 25, 30, ..., 100

n_pairs = len(list(EMA_FAST_VALUES)) * len(list(EMA_SLOW_VALUES))
print(f"{n_pairs} parameter combinations to evaluate")

221 parameter combinations to evaluate


## 3. Run sweep

For each config: build a new frozen `StrategyConfig` via `dataclasses.replace`, run the backtester, record metrics + per-exit-reason counts + average trade duration (minutes).

In [6]:
rows = []
for fast in EMA_FAST_VALUES:
    for slow in EMA_SLOW_VALUES:
        cfg = dataclasses.replace(StrategyConfig(), ema_fast=fast, ema_slow=slow)
        result = Backtester(InverseEMACrossoverStrategy(cfg), symbol=SYMBOL).run(df, interval=INTERVAL)

        reason_counts = Counter(
            t.exit_reason.value if t.exit_reason else "unknown"
            for t in result.trades
        )
        durations_min = [
            t.duration.total_seconds() / 60
            for t in result.trades if t.duration is not None
        ]
        avg_duration_min = sum(durations_min) / len(durations_min) if durations_min else 0.0

        row = {
            "ema_fast":         fast,
            "ema_slow":         slow,
            "trades":           result.total_trades,
            "win_rate":         result.win_rate,
            "total_pnl_bps":    result.total_pnl_bps,
            "profit_factor":    result.profit_factor,
            "max_dd_bps":       result.max_drawdown_bps,
            "sharpe":           result.sharpe_approx,
            "avg_duration_min": avg_duration_min,
        }
        # One column per exit reason (zero-filled when a reason didn't fire)
        for reason in ExitReason:
            row[f"n_{reason.value}"] = reason_counts.get(reason.value, 0)
        rows.append(row)

results = pd.DataFrame(rows)
print(f"Evaluated {len(results)} configurations")

Evaluated 221 configurations


## 4. Top results

Filter out configs with too few trades before ranking — a high Sharpe on 3 trades is noise, not signal.

In [9]:
MIN_TRADES = 20

top = (
    results[results.trades >= MIN_TRADES]
    .sort_values("total_pnl_bps", ascending=False)
    .head(10)
)
top

,ema_fast,ema_slow,trades,win_rate,total_pnl_bps,profit_factor,max_dd_bps,sharpe,avg_duration_min,n_stop_loss,n_take_profit,n_trailing_stop,n_signal_flip,n_time_stop,n_invalidation,n_force_close
47,9,85,55,0.472727,-202.961370,0.865516,413.406944,-0.056366,131.181818,0,0,28,27,0,0,0
46,9,80,56,0.482143,-214.219570,0.861422,375.673839,-0.057718,133.660714,0,0,28,28,0,0,0
48,9,90,53,0.452830,-321.516771,0.796454,436.174490,-0.090068,133.018868,0,0,30,23,0,0,0
63,11,80,52,0.442308,-322.716612,0.785045,412.352834,-0.097297,147.692308,0,0,28,24,0,0,0
205,29,25,52,0.230769,-345.427920,0.817531,850.650670,-0.065125,185.769231,0,0,43,9,0,0,0
71,13,35,62,0.435484,-349.836122,0.805952,382.569882,-0.085400,140.564516,0,0,36,26,0,0,0
218,29,90,34,0.382353,-360.371088,0.673451,426.906369,-0.154541,190.588235,0,0,26,8,0,0,0
220,29,100,36,0.388889,-372.341986,0.666981,459.603360,-0.154956,193.750000,0,0,26,10,0,0,0
79,13,75,49,0.448980,-375.497027,0.735157,382.481160,-0.124394,156.122449,0,0,28,21,0,0,0
196,27,65,39,0.384615,-375.736561,0.700412,376.269143,-0.147367,174.230769,0,0,28,11,0,0,0


## 5. Heatmaps

Each cell = one `(ema_fast, ema_slow)` configuration. Look for *regions* of green — a bright cell surrounded by red is usually an overfit outlier; a bright cell with bright neighbours is a robust parameter choice.

In [10]:
grid = results.pivot(index="ema_fast", columns="ema_slow", values="sharpe")
fig = px.imshow(
    grid,
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    aspect="auto",
    labels=dict(x="ema_slow", y="ema_fast", color="Sharpe"),
    title=f"Sharpe (inverse EMA) — {SYMBOL} {INTERVAL}m ({START_TIME} → {END_TIME or 'now'})",
)
fig.show()

In [11]:
grid = results.pivot(index="ema_fast", columns="ema_slow", values="total_pnl_bps")
fig = px.imshow(
    grid,
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    aspect="auto",
    labels=dict(x="ema_slow", y="ema_fast", color="Total P&L (bps)"),
    title=f"Total P&L (inverse EMA) — {SYMBOL} {INTERVAL}m",
)
fig.show()

## 6. Exit-reason distribution for the top config

Strategy quality isn't just about PnL — *how* trades close matters. A strategy whose winners all come from `take_profit` behaves very differently from one relying on `signal_flip`.

In [12]:
best = top.iloc[0]
exit_cols = [c for c in results.columns if c.startswith("n_")]
exit_breakdown = best[exit_cols].rename(lambda c: c.removeprefix("n_"))
exit_breakdown = exit_breakdown[exit_breakdown > 0]

fig = px.bar(
    x=exit_breakdown.index,
    y=exit_breakdown.values,
    labels=dict(x="exit reason", y="trade count"),
    title=f"Exit reasons — best inverse EMA config (ema_fast={int(best.ema_fast)}, ema_slow={int(best.ema_slow)})",
)
fig.show()